In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

crime = spark.table("scottish_housing_ws.silver.crime")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
crime.printSchema()
crime.show(3)

In [0]:
crime.select("crime_type").distinct().orderBy("crime_type").show(20, truncate=False)

In [0]:
# Filter crime to council areas and pivot crime types 
# Keep All Crimes as the headline figure plus the group breakdowns
# Exclude All Offences -- that's a separate category from All Crimes
# Pivot so each crime type becomes a column -- one row per council per year

crime_councils = (
    crime
    .filter(F.col("area_type") == "Council Area")
    .filter(F.col("crime_type") != "All Offences")
)

crime_pivoted = (
    crime_councils
    .groupBy("fiscal_year_start", "area_code", "area_name")
    .pivot("crime_type", [
        "All Crimes",
        "All Group 1: Non-sexual crimes of violence",
        "All Group 2: Sexual crimes",
        "All Group 3: Crimes of dishonesty",
        "All Group 4: Damage and reckless behaviour",
        "All Group 5: Crimes against society",
        "All Group 6: Antisocial offences",
        "All Group 7: Miscellaneous offences",
        "All Group 8: Road traffic offences"
    ])
    .agg(F.first("crimes_recorded"))
    .withColumnRenamed("All Crimes", "all_crimes")
    .withColumnRenamed("All Group 1: Non-sexual crimes of violence", "group1_violence")
    .withColumnRenamed("All Group 2: Sexual crimes", "group2_sexual")
    .withColumnRenamed("All Group 3: Crimes of dishonesty", "group3_dishonesty")
    .withColumnRenamed("All Group 4: Damage and reckless behaviour", "group4_damage")
    .withColumnRenamed("All Group 5: Crimes against society", "group5_society")
    .withColumnRenamed("All Group 6: Antisocial offences", "group6_antisocial")
    .withColumnRenamed("All Group 7: Miscellaneous offences", "group7_miscellaneous")
    .withColumnRenamed("All Group 8: Road traffic offences", "group8_road_traffic")
)

In [0]:
hpi_annual = (
    hpi
    .filter(F.col("area_code").startswith("S12"))
    .withColumn("year", F.col("year_month").substr(1, 4).cast("int"))
    .groupBy("year", "area_code")
    .agg(
        F.round(F.avg("average_price"), 0).alias("avg_price_calendar_year"),
        F.sum("sales_volume").alias("total_sales_volume")
    )
)

In [0]:
gold = (
    crime_pivoted
    .join(
        hpi_annual,
        on=[
            crime_pivoted.area_code == hpi_annual.area_code,
            crime_pivoted.fiscal_year_start == hpi_annual.year
        ],
        how="inner"
    )
    .select(
        "fiscal_year_start",
        crime_pivoted.area_code,
        "area_name",
        "avg_price_calendar_year",
        "total_sales_volume",
        "all_crimes",
        "group1_violence",
        "group2_sexual",
        "group3_dishonesty",
        "group4_damage",
        "group5_society",
        "group6_antisocial",
        "group7_miscellaneous",
        "group8_road_traffic"
    )
    .orderBy("area_code", "fiscal_year_start")
)

In [0]:
print(f"Row count: {gold.count()}")

# Edinburgh over time
gold.filter(F.col("area_code") == "S12000036").orderBy("fiscal_year_start").show(5)
gold.filter(F.col("area_code") == "S12000036").orderBy(F.col("fiscal_year_start").desc()).show(5)

In [0]:

(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.crime_rate_vs_house_price")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_crime_rate_vs_house_price/")
)

print("Gold 13 complete.")

In [0]:
# Crime Rate vs House Price
# Sources: silver.crime, silver.uk_hpi
# Grain: annual, by council area
# Join keys: area_code + year

# Crime data is fiscal year granularity (April to March).
# HPI is monthly. We aggregate HPI to calendar year average price as an approximation -- fiscal year and calendar year are close enough for this analysis given crime data is already a coarse annual aggregate.

# Crime types are group-level aggregates only (subcategories dropped in silver)
# Coverage overlap: crime 1996/97 to 2023/24, HPI from 1968
# Effective range after join: 1996 to 2023